In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [2]:
df = pd.read_csv(
    "../dataset/indian_railway_delay_dataset_25000_unclean.csv",
    encoding="latin1",
    on_bad_lines="skip"
)

df.head()

,train_no,train_name,zone,source_station,destination_station,current_station,day_of_week,month,festival,weather,...,track_congestion,station_congestion,previous_station_delay,average_route_delay,distance_from_source,distance_remaining,train_priority,locomotive_type,maintenance_flag,delay_minutes
0,12301,Howrah Rajdhani,ER,NDLS,PRYJ,CNB,Friday,5,NaN,Clear,...,Medium,Low,18,9,1657,1780,Rajdhani,WAP-5,No,19
1,12622,Tamil Nadu Express,SR,PRYJ,LKO,CNB,Sunday,6,NaN,Cloudy,...,Medium,High,9,10,1182,393,Vande Bharat,WAP-7,Yes,29
2,22436,Vande Bharat,NR,PRYJ,BCT,CNB,Wednesday,11,NaN,Cloudy,...,Medium,Low,1,0,1409,1140,Express,WDP-4D,Yes,0
3,22436,Vande Bharat,NR,NDLS,KOTA,AGC,Friday,7,NaN,Clear,...,Medium,Medium,22,0,741,449,Express,WAG-9,Yes,14
4,12951,Mumbai Rajdhani,WR,BCT,LKO,MAS,Wednesday,2,NaN,Clear,...,Medium,High,9,18,601,890,Express,WAG-9,Yes,29


In [3]:
df = df[
    [
        "train_no",
        "weather",
        "day_of_week",
        "distance_from_source",
        "previous_station_delay",
        "track_congestion",
        "station_congestion",
        "delay_minutes",
    ]
]

df.head()

,train_no,weather,day_of_week,distance_from_source,previous_station_delay,track_congestion,station_congestion,delay_minutes
0,12301,Clear,Friday,1657,18,Medium,Low,19
1,12622,Cloudy,Sunday,1182,9,Medium,High,29
2,22436,Cloudy,Wednesday,1409,1,Medium,Low,0
3,22436,Clear,Friday,741,22,Medium,Medium,14
4,12951,Clear,Wednesday,601,9,Medium,High,29


In [4]:
df.isnull().sum()

train_no                  0
weather                   0
day_of_week               0
distance_from_source      0
previous_station_delay    0
track_congestion          0
station_congestion        0
delay_minutes             0
dtype: int64

In [5]:
from sklearn.preprocessing import LabelEncoder

encoders = {}

categorical_columns = [
    "weather",
    "day_of_week",
    "track_congestion",
    "station_congestion"
]

for column in categorical_columns:
    encoder = LabelEncoder()
    df[column] = encoder.fit_transform(df[column])
    encoders[column] = encoder

df.head()

,train_no,weather,day_of_week,distance_from_source,previous_station_delay,track_congestion,station_congestion,delay_minutes
0,12301,0,0,1657,18,2,1,19
1,12622,1,3,1182,9,2,0,29
2,22436,1,6,1409,1,2,1,0
3,22436,0,0,741,22,2,2,14
4,12951,0,6,601,9,2,0,29


In [6]:
X = df.drop("delay_minutes", axis=1)

y = df["delay_minutes"]

print(X.shape)
print(y.shape)

(25000, 7)
(25000,)


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(20000, 7)
(5000, 7)


In [12]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=20,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

print("✅ Model trained!")

✅ Model trained!


In [13]:
predictions = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, predictions))
print("R² Score:", r2_score(y_test, predictions))

MAE: 7.696581455389631
R² Score: 0.6766686760488001


In [14]:
import joblib

joblib.dump(model, "../ml/railway_delay_model.pkl")
joblib.dump(encoders, "../ml/label_encoders.pkl")

print("✅ Model and encoders saved!")

✅ Model and encoders saved!


In [15]:
import os

print("Model size:", os.path.getsize("../ml/railway_delay_model.pkl"), "bytes")

Model size: 7206209 bytes


In [16]:
print(X.columns.tolist())

['train_no', 'weather', 'day_of_week', 'distance_from_source', 'previous_station_delay', 'track_congestion', 'station_congestion']
